# Reproducibility Example
**台灣 YouTube 核能議題：LLM 自動化標注示範**

本 notebook 示範如何使用本 repo 的 prompts 與 dictionaries，對 YouTube 留言進行：
1. 文本預處理（清理 + 核能關鍵字篩選）
2. 框架標注（9 類論述框架）
3. 不文明標注（二元分類）

**資料**：使用 5 筆示範留言，無需 API key 即可理解流程。實際執行需要 OpenAI 或 Anthropic API key。

## 0. 安裝套件

In [ ]:
# !pip install openai anthropic pandas

## 1. 文本預處理

In [ ]:
import pandas as pd
import re

# 示範留言（5 筆）
sample_comments = [
    {"id": "c001", "text": "核電安全有問題，福島事件還不夠教訓嗎？台灣這麼小根本承受不起核災"},
    {"id": "c002", "text": "核電成本最低，廢核電費一定大漲，民進黨在害台灣產業競爭力！"},
    {"id": "c003", "text": "冥禁黨廢核是政治算計，根本不管人民死活，一群垃圾"},
    {"id": "c004", "text": "SMR小型核電廠技術已經很成熟，核廢料也有玻璃固化技術可以處理"},
    {"id": "c005", "text": "這次公投大家一定要去投票，不管支持或反對，讓公民的聲音被聽到"},
]

df = pd.DataFrame(sample_comments)
print(f"原始資料：{len(df)} 筆")
df

In [ ]:
# 載入核能關鍵字庫
def load_keywords(path="../dictionaries/nuclear_keywords.txt"):
    keywords = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                keywords.append(line)
    return keywords

# 文本清理
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"@\S+", "", text)          # 移除 @mention
    text = re.sub(r"https?://\S+", "", text)  # 移除網址
    text = re.sub(r"\s+", " ", text).strip()  # 壓縮空白
    return text

# 核能主題篩選
def is_nuclear_related(text, keywords):
    pattern = "|".join(keywords)
    return bool(re.search(pattern, text))

df["clean_text"] = df["text"].apply(clean_text)

# 示範：直接用部分關鍵字篩選
sample_keywords = ["核", "能源", "公投", "電費", "廢核", "福島", "台電"]
df["is_nuclear"] = df["clean_text"].apply(lambda t: is_nuclear_related(t, sample_keywords))

print(f"主題篩選後：{df['is_nuclear'].sum()} 筆")
df[["id", "clean_text", "is_nuclear"]]

## 2. 框架標注（Frame Annotation）

使用 GPT-4.1-mini 進行留言框架標注。提示詞來源：`prompts/comment_frame_prompt.md`

In [ ]:
import json

FRAME_SYSTEM_PROMPT = """你是專業的媒體框架分析研究員，專精於台灣核能議題的論述框架分析。你的任務是根據下方框架定義，對 YouTube 的影片內容進行框架標注。

【框架定義】
1. 經濟發展與供電穩定：電價、供電穩定、產業缺電風險
2. 低碳永續：氣候變遷、減碳、淨零排放
3. 失控科技與代價負擔：核災、輻射危害、核廢料選址、世代正義
4. 技術發展與可行性：新型核能技術、引用資料反駁安全疑慮
5. 權衡取捨：兩種以上能源的比較
6. 公共問責：政黨政府決策責任、承諾跳票
7. 公眾參與：公投、投票行為、立場表態
8. 能源自主：國家安全、能源獨立、兩岸議題
9. 無關/無意義/不明確：閒聊、過場、無法判定

【標注規則】
- 只能選出1個「最主導」的框架
- 框架名稱必須完全符合上列 9 個之一
- 信心度（confidence）：高（框架明確）/ 中（有跡可循）/ 低（語意不明）

【輸出格式】
只輸出 JSON，不要任何解釋：
{"primary_frame": "框架名稱", "confidence": "高/中/低"}"""

def annotate_frame_openai(text, client):
    """使用 GPT-4.1-mini 進行框架標注"""
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": FRAME_SYSTEM_PROMPT},
            {"role": "user", "content": text}
        ],
        temperature=0,
        max_tokens=100,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

# 示範預期輸出（無需 API key）
expected_frame_output = [
    {"id": "c001", "text": "核電安全有問題...", "primary_frame": "失控科技與代價負擔", "confidence": "高"},
    {"id": "c002", "text": "核電成本最低...",   "primary_frame": "經濟發展與供電穩定", "confidence": "高"},
    {"id": "c003", "text": "冥禁黨廢核...",     "primary_frame": "公共問責",           "confidence": "高"},
    {"id": "c004", "text": "SMR小型核電廠...",  "primary_frame": "技術發展與可行性",    "confidence": "高"},
    {"id": "c005", "text": "這次公投...",        "primary_frame": "公眾參與",           "confidence": "高"},
]

pd.DataFrame(expected_frame_output)

## 3. 不文明標注（Incivility Annotation）

使用 GPT-4.1-mini 進行不文明二元標注。提示詞來源：`prompts/incivility_prompt.md`

In [ ]:
INCIVILITY_PROMPT = """你是一位專門研究台灣社群媒體與政治溝通的內容標記研究員。

【任務】
以下是台灣核能主題的YouTube影片留言，請獨立判斷每則留言是否包含「不文明言論」，
請務必先在 reason 欄位進行邏輯分析，後才決定 label。

【不文明言論判定標準】
只要出現以下任一情況，即標記為 1：

*違反禮貌規範*
辱罵或羞辱、粗俗用語、髒話、諷刺或反諷、人身攻擊、負面標籤稱呼、
輕蔑語氣、貶低特定對象或群體、質疑智力、攻擊名譽

*違反民主規範*
煽動暴力、挑釁、歧視、威脅、宣稱政府無能、體制崩壞、治理失敗

否則標記為 0。

【輸出要求】
- 僅輸出 JSON
- 不得新增欄位

格式範例：
{"results": [{"id": "c001", "reason": "判定理由20字內", "label": 0}]}"""

def annotate_incivility_batch(comments_batch, client):
    """批次不文明標注（10-20 筆/次）"""
    user_content = json.dumps(
        [{"id": c["id"], "text": c["clean_text"]} for c in comments_batch],
        ensure_ascii=False
    )
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": INCIVILITY_PROMPT},
            {"role": "user", "content": user_content}
        ],
        temperature=0,
        max_tokens=500,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)["results"]

# 示範預期輸出（無需 API key）
expected_incivility_output = [
    {"id": "c001", "reason": "批評核電政策，無人身攻擊",   "label": 0},
    {"id": "c002", "reason": "批評政黨政策，有輕蔑語氣",   "label": 1},
    {"id": "c003", "reason": "冥禁黨為負面標籤，含辱罵詞", "label": 1},
    {"id": "c004", "reason": "技術性討論，語氣中性",       "label": 0},
    {"id": "c005", "reason": "號召投票，無不文明",         "label": 0},
]

pd.DataFrame(expected_incivility_output)

## 4. 信度評估示範

使用 Cohen's κ 評估人工標注與 LLM 標注的一致性。

In [ ]:
from sklearn.metrics import cohen_kappa_score, classification_report

# 示範：人工標注 vs LLM 標注
human_labels   = [0, 1, 1, 0, 0]  # 人工標注
llm_labels     = [0, 1, 1, 0, 0]  # LLM 標注（示範用，與人工一致）

kappa = cohen_kappa_score(human_labels, llm_labels)
print(f"Cohen's κ = {kappa:.3f}")
print()
print(classification_report(
    human_labels, llm_labels,
    target_names=["文明(0)", "不文明(1)"]
))

# 本研究實際信度結果（供參考）
print("\n=== 本研究實際信度 ===")
print("影片立場（human-claude）: κ = 0.85")
print("影片框架高信心子集: κ = 0.79")
print("留言框架高信心子集: κ = 0.83")
print("不文明標注最佳策略: κ = 0.701")

## 延伸資源

- **Prompts**：`prompts/` 資料夾包含完整提示詞與迭代紀錄
- **Codebooks**：`codebooks/` 包含框架定義與不文明操作型定義
- **Dictionaries**：`dictionaries/` 包含關鍵字庫、停用詞、誤辨修正表

**引用**：  
Lee, Y.-J. (2026). *Nuclear Energy Controversy on YouTube: Frame Construction, Public Response, and Incivility Dynamics in Taiwan*. Master's Thesis, Graduate Institute of Journalism, National Taiwan University.